# Kiểm tra và benchmark truy hồi RAG — Huế

Notebook này làm hai việc:

1. Thử tay luồng truy hồi: embed chunk, tìm theo câu hỏi, xem ngữ cảnh đưa vào LLM.
2. Chạy benchmark truy hồi trên bộ câu hỏi gán nhãn `eval/rag_queries.yml`, so dense (bge-m3) với BM25 và random.

Chuẩn bị (chạy trong terminal, ở thư mục gốc repo):

```
docker start <container postgres>        # DB ở localhost:5433
pip install -e .[embed]                  # sentence-transformers + torch
python -m pipeline migrate               # có migration 007 (cột embedding_model)
python -m pipeline download-model        # tải bge-m3 một lần (~4.5 GB tính cả bản safetensors)
python -m pipeline embed
```

In [ ]:
from IPython.display import Markdown, display

from pipeline import db
from pipeline.embed import get_embedder, run_embed_chunks
from rag import HybridRetriever, RAGRetriever
from rag import benchmark

conn = db.connect()
print("Database:", conn.info.dbname)

## 1. Trạng thái index

Mỗi chunk ghi lại model đã sinh embedding. Retriever chỉ tìm trên chunk cùng model với query.

In [ ]:
with conn.cursor() as cur:
    cur.execute(
        "SELECT COALESCE(embedding_model, '(chưa embed)'), COUNT(*)"
        " FROM place_chunks GROUP BY 1 ORDER BY 2 DESC"
    )
    for model, n in cur.fetchall():
        print(f"{model:30} {n:6d} chunk")

## 2. Sinh embedding

`get_embedder()` đọc `EMBEDDING_MODEL`, mặc định `BAAI/bge-m3`. Không dùng `DeterministicEmbedder` ở đây: vector của nó là giá trị băm, không mang nghĩa, nên mọi kết quả tìm kiếm sẽ là ngẫu nhiên.

Chỉ chunk chưa có embedding của model hiện tại mới được sinh. Thêm `force=True` để sinh lại tất cả.

In [ ]:
embedder = get_embedder()
count = run_embed_chunks(conn, embedder=embedder)
print(f"Đã sinh embedding cho {count} chunk bằng {embedder.model_name}")

In [ ]:
retriever = RAGRetriever(conn, embedder=embedder)
total, embedded = retriever.index_status()
print(f"{embedded}/{total} chunk có embedding của {embedder.model_name}")

# Retriever agent dùng: dense + BM25 (giữ dấu, bỏ dấu, bigram bỏ dấu), ghép bằng RRF.
# Mặc định loại chunk của category không phải điểm đến (boi_canh, ha_tang). Dùng lại model đã tải.
hybrid = HybridRetriever.from_connection(conn, embedder=embedder)
print(hybrid.name)

## 3. Thử truy vấn

`hybrid` là retriever agent dùng. Với hybrid, `similarity` là điểm RRF (tổng 1/(60 + hạng) qua hai nhánh), không phải cosine.

In [ ]:
def show(results, width=200):
    for i, c in enumerate(results, 1):
        print(f"[{i}] {c.place_name} ({c.category}) — similarity {c.similarity:.4f}")
        print(f"    {c.content[:width]}...")
        print(f"    Nguồn: {c.source_url}\n")


show(hybrid.search("Lịch sử và kiến trúc chùa Thiên Mụ", top_k=3))

In [ ]:
# Câu mô tả không nêu tên địa danh: dense phải hiểu nghĩa, BM25 thường trượt.
show(hybrid.search("Lăng của vị vua làm nhiều thơ, có hồ Lưu Khiêm", top_k=3))

In [ ]:
# Gõ không dấu: dense trượt, nhánh BM25 kéo địa danh đúng lên.
query = "lang tu duc co gi dep"
print("--- dense")
show(retriever.search(query, top_k=3), width=100)
print("--- hybrid")
show(hybrid.search(query, top_k=3), width=100)

In [ ]:
# Ngữ cảnh sẵn sàng đưa vào prompt của agent.
print(hybrid.build_rag_context(hybrid.search("Chùa Thiên Mụ được xây năm nào?", top_k=3)))

## 4. Benchmark truy hồi

Nhãn gán ở mức địa danh (QID Wikidata). Mọi chunk của một địa danh đúng được tính là đúng. Chỉ số chạy trên top-k chunk mà agent sẽ đưa vào prompt:

| Chỉ số | Ý nghĩa | Vì sao cần |
|---|---|---|
| Hit@k | Top-k có ít nhất một chunk của địa danh đúng | Chỉ số chính: LLM có thấy được nguồn đúng không |
| MRR@10 | Trung bình 1/hạng của chunk đúng đầu tiên | Chunk đúng nằm càng cao càng tốt |
| nDCG@5 | Chất lượng thứ hạng, tính mọi chunk đúng trong top-5 | Phân biệt "1 chunk đúng ở hạng 5" với "5 chunk đúng" |
| Precision@5 | Tỷ lệ chunk đúng trong 5 chunk đưa vào prompt | Chunk sai là nhiễu, dễ làm LLM trả lời sai |
| AnswerHit@5 | Top-5 có chunk thuộc địa danh đúng *và* chứa từ khoá câu trả lời | Đúng địa danh chưa đủ, phải đúng đoạn chứa đáp án |
| Recall@k (câu `multi`) | Tỷ lệ địa danh đúng xuất hiện trong top-k | Câu "các lăng vua Nguyễn" cần nhiều địa danh |
| Latency p50/p95 | Thời gian embed query + truy vấn SQL | Agent gọi tool này trong mỗi lượt hỏi đáp |

Retriever đứng đầu là **hybrid**, so với: **hybrid_fold** (bản hybrid chỉ bỏ dấu), **dense** và **bm25_multi** (từng nhánh riêng lẻ, tức ablation), **BM25** giữ dấu, **bm25_fold**, và **random** (mức sàn). Chạy thêm `python -m rag.benchmark --all-places` để đo tác động của việc loại category không phải điểm đến. Chênh lệch giữa hai retriever kèm khoảng tin cậy bootstrap 95%; khoảng không chứa 0 thì chênh lệch đáng tin với cỡ bộ câu hỏi hiện tại.

Câu hỏi mà địa danh đúng không có chunk nào bị loại khỏi số liệu và liệt kê riêng ở cuối báo cáo: đó là lỗ hổng dữ liệu, không phải lỗi truy hồi.

In [ ]:
report, results = benchmark.run(conn, embedder=embedder, out_dir=None)
display(Markdown(report))

## 5. Phân tích lỗi: hybrid so với dense theo từng câu

In [ ]:
by_query = {
    name: {r.query.id: r for r in rs} for name, rs in results.items()
}
a_name, b_name = hybrid.name, retriever.name
print(f"{'id':6} {'type':14} {'hybrid':>6} {'dense':>6}  query")
for qid, a in by_query[a_name].items():
    b = by_query[b_name][qid]
    if a.metrics["mrr"] != b.metrics["mrr"]:
        print(f"{qid:6} {a.query.type:14} {a.metrics['mrr']:6.2f} {b.metrics['mrr']:6.2f}  {a.query.query}")

In [ ]:
# Xem top-5 của một câu cụ thể.
qid = "q005"
for name in results:
    r = by_query[name][qid]
    print(f"--- {name}: hit@5={r.metrics['hit@5']:.0f}, mrr={r.metrics['mrr']:.2f}")
    for c in r.chunks[:5]:
        print(f"    {c.place_name}: {c.content[:90]}...")

## 6. Lưu báo cáo

Ghi `data/qa/rag_benchmark_<ngày>.md` và `.csv` (từng câu, từng retriever) để đưa vào báo cáo đồ án. Terminal tương đương: `python -m rag.benchmark`.

In [ ]:
report, _ = benchmark.run(conn, embedder=embedder)
print("Đã ghi data/qa/rag_benchmark_*.md và .csv")

In [ ]:
conn.close()